# Local Pipeline — Final Overnight Run (Priority-Ordered)
Every experiment section below uses the same resumable, chunked pattern: if this notebook is interrupted at any point (session drop, crash, power loss) and re-run from the top, already-completed chunks are detected and skipped automatically — no work is redone. Sections are ordered by priority, not by section number, so a partial overnight run still produces the most important results first. Run top to bottom; if time runs out partway through, whatever finished is complete and usable.

In [1]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    level = root.replace("/kaggle/input", "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root) or root}/")
    if level < 5:  # don't go too deep once we've found real files
        for f in files[:5]:
            print(f"{indent}  {f}")
        if len(files) > 5:
            print(f"{indent}  ... ({len(files)} files total)")

input/
  notebooks/
    asmaakther/
      final-notebook-v3/
        sam_vit_b_01ec64.pth
        __notebook__.ipynb
        __output__.json
        results/
          dinov2_corruption_final.csv
          clip_localization_diagnostic.csv
          severity_sweep_final.csv
          dinov2_rescore_results.csv
          priority2_clip_diagnostic/
          optional_main_experiment/
          priority4_dinov2_clean/
          priority1_dinov2_corruption/
          priority3_severity_sweep/


## 1. Setup — Installs, Device Check, Model Loads, Dataset

In [2]:
import sys, subprocess

def pip_install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

pip_install("git+https://github.com/openai/CLIP.git")
pip_install("git+https://github.com/facebookresearch/segment-anything.git")
pip_install("transformers")
pip_install("datasets")
pip_install("pycocotools")
pip_install("torchvision")
pip_install("opencv-python-headless")

print("Packages installed.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.8 MB/s eta 0:00:00
Packages installed.


In [3]:
import torch, clip, time, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageFilter, ImageEnhance
from pycocotools import mask as maskUtils
from datasets import load_dataset
from segment_anything import sam_model_registry, SamPredictor
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
import torchvision.transforms as T

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
if device == "cpu":
    print("WARNING: no GPU detected. This pipeline will be very slow on CPU.")

RESULTS_DIR = "./results"
os.makedirs(RESULTS_DIR, exist_ok=True)
import shutil

PREVIOUS_OUTPUT_DIR = "/kaggle/input/notebooks/asmaakther/final-notebook-v3/results"

if os.path.exists(PREVIOUS_OUTPUT_DIR):
    shutil.copytree(PREVIOUS_OUTPUT_DIR, RESULTS_DIR, dirs_exist_ok=True)
    print("Restored previous checkpoints into", RESULTS_DIR)
else:
    print("No previous checkpoint folder found — starting fresh.")

# Reproducibility: fixes the random seed so Gaussian noise corruption is
# deterministic across runs/sessions, not different each time.
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
print(f"Random seed set to {SEED}.")


Device: cuda
Restored previous checkpoints into ./results
Random seed set to 42.


In [4]:
model, preprocess = clip.load("ViT-B/32", device=device)
print("CLIP loaded.")


100%|████████████████████████████████████████| 338M/338M [00:01<00:00, 282MiB/s]


CLIP loaded.


In [5]:
import urllib.request

SAM_CKPT = "sam_vit_b_01ec64.pth"
SAM_URL = "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth"

if not os.path.exists(SAM_CKPT):
    print("Downloading SAM checkpoint (~375MB)...")
    urllib.request.urlretrieve(SAM_URL, SAM_CKPT)
    print("Download complete.")
else:
    print("SAM checkpoint already present, skipping download.")

sam = sam_model_registry["vit_b"](checkpoint=SAM_CKPT).to(device)
predictor = SamPredictor(sam)
print("SAM loaded.")


Download complete.
SAM loaded.


In [6]:
gd_model_id = "IDEA-Research/grounding-dino-tiny"
gd_processor = AutoProcessor.from_pretrained(gd_model_id)
gd_model = AutoModelForZeroShotObjectDetection.from_pretrained(gd_model_id).to(device)
gd_model.eval()
print("Grounding DINO loaded.")


preprocessor_config.json:   0%|          | 0.00/457 [00:00<?, ?B/s]

The image processor of type `GroundingDinoImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

added_tokens.json:   0%|          | 0.00/82.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/689M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/990 [00:00<?, ?it/s]

Grounding DINO loaded.


In [7]:
dinov2 = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14').to(device)
dinov2.eval()
dinov2_transform = T.Compose([
    T.Resize((518, 518)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
print("DINOv2 loaded.")


Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip


/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vits14/dinov2_vits14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vits14_pretrain.pth


100%|██████████| 84.2M/84.2M [00:00<00:00, 521MB/s]


DINOv2 loaded.


In [8]:
dataset = load_dataset("moondream/refcoco-m")
print("Dataset loaded. Validation images:", len(dataset["validation"]))


README.md: 0.00B [00:00, ?B/s]

data/validation-00000-of-00002.parquet:   0%|          | 0.00/308M [00:00<?, ?B/s]

data/validation-00001-of-00002.parquet:   0%|          | 0.00/302M [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/1190 [00:00<?, ? examples/s]

Dataset loaded. Validation images: 1190


### 1a. Generic Resumable-Chunk Runner
Used by every experiment section below. Splits work into fixed-size chunks of images; each chunk is saved to its own CSV as soon as it finishes. On re-run, any chunk whose file already exists is skipped entirely — this is what makes every section below safe to interrupt.

In [9]:
def run_chunked(name, total_images, chunk_size, checkpoint_dir, process_image_fn):
    """
    Generic resumable runner. process_image_fn(img_idx) must return a list
    of row dicts for that image (can be empty). Returns the combined
    DataFrame across all chunks (existing + newly computed).
    """
    os.makedirs(checkpoint_dir, exist_ok=True)
    all_chunks = []

    for chunk_start in range(0, total_images, chunk_size):
        chunk_end = min(chunk_start + chunk_size, total_images)
        chunk_path = f"{checkpoint_dir}/{name}_{chunk_start}_{chunk_end}.csv"

        if os.path.exists(chunk_path):
            print(f"[{name}] chunk {chunk_start}-{chunk_end} already done, skipping", flush=True)
            all_chunks.append(pd.read_csv(chunk_path))
            continue

        chunk_rows = []
        t0 = time.time()
        for img_idx in range(chunk_start, chunk_end):
            try:
                chunk_rows.extend(process_image_fn(img_idx))
            except Exception as e:
                print(f"  ERROR on image {img_idx}: {e}", flush=True)

        chunk_df = pd.DataFrame(chunk_rows)
        chunk_df.to_csv(chunk_path, index=False)
        all_chunks.append(chunk_df)
        print(f"[{name}] chunk {chunk_start}-{chunk_end} done in {time.time()-t0:.0f}s, saved", flush=True)

    return pd.concat(all_chunks, ignore_index=True) if all_chunks else pd.DataFrame()

print("Resumable chunk runner loaded.")


Resumable chunk runner loaded.


## 2. Canonical CLIP + SAM Pipeline (Baseline / Ablation)

In [10]:
def get_patch_features(pil_image):
    activation = {}
    def capture_attn_input(module, input, output):
        activation["input"] = input
    last_block = model.visual.transformer.resblocks[-1]
    handle = last_block.attn.register_forward_hook(capture_attn_input)
    image_input = preprocess(pil_image).unsqueeze(0).to(device)
    with torch.no_grad():
        _ = model.encode_image(image_input)
    handle.remove()
    x = activation["input"][0]
    attn = last_block.attn
    embed_dim = attn.embed_dim
    W_v = attn.in_proj_weight[2 * embed_dim:]
    b_v = attn.in_proj_bias[2 * embed_dim:]
    V = torch.nn.functional.linear(x, W_v, b_v)
    V_proj = V.squeeze(1) @ model.visual.proj
    V_proj = torch.nn.functional.normalize(V_proj, dim=-1)
    return V_proj[1:]

def get_clip_heatmap(image, prompt):
    patch_features = get_patch_features(image)
    text = clip.tokenize([prompt]).to(device)
    with torch.no_grad():
        text_features = model.encode_text(text)
        text_features = text_features / text_features.norm(dim=-1, keepdim=True)
    similarity = patch_features @ text_features.T
    return similarity.reshape(7, 7).detach().cpu()

def heatmap_to_points(image, heatmap, grid_size, top_k=5, max_dist_frac=0.35):
    topk_values, topk_indices = torch.topk(heatmap.flatten(), k=top_k)
    raw_points = []
    for idx in topk_indices:
        idx = idx.item()
        row, col = idx // grid_size, idx % grid_size
        x = (col + 0.5) * image.width / grid_size
        y = (row + 0.5) * image.height / grid_size
        raw_points.append([x, y])
    raw_points = np.array(raw_points)
    median_pt = np.median(raw_points, axis=0)
    diag = np.hypot(image.width, image.height)
    dists = np.linalg.norm(raw_points - median_pt, axis=1)
    keep = dists <= (max_dist_frac * diag)
    if keep.sum() == 0:
        keep = np.array([True] + [False] * (len(raw_points) - 1))
    return raw_points[keep], raw_points, keep

def points_to_box(points, image, padding_frac=0.15):
    x_min, y_min = points.min(axis=0)
    x_max, y_max = points.max(axis=0)
    box_w, box_h = x_max - x_min, y_max - y_min
    pad_x = max(box_w * padding_frac, image.width * 0.03)
    pad_y = max(box_h * padding_frac, image.height * 0.03)
    x_min = max(0, x_min - pad_x); y_min = max(0, y_min - pad_y)
    x_max = min(image.width, x_max + pad_x); y_max = min(image.height, y_max + pad_y)
    return np.array([x_min, y_min, x_max, y_max])

def score_masks_against_gt(masks, gt_mask):
    gt = gt_mask.astype(bool)
    ious = []
    for m in masks:
        pred = m.astype(bool)
        intersection = np.logical_and(pred, gt).sum()
        union = np.logical_or(pred, gt).sum()
        ious.append(intersection / union if union > 0 else 0.0)
    return ious

def run_clip_sam(image, prompt, gt_mask, predictor, top_k=5, max_dist_frac=0.35):
    clip_heatmap = get_clip_heatmap(image, prompt)
    points, raw_points, keep_mask = heatmap_to_points(image, clip_heatmap, 7, top_k, max_dist_frac)
    box = points_to_box(points, image)
    labels = np.ones(len(points), dtype=np.int32)
    predictor.set_image(np.array(image))
    masks, sam_scores, logits = predictor.predict(box=box, point_coords=points.astype(np.float32),
                                                     point_labels=labels, multimask_output=True)
    ious = score_masks_against_gt(masks, gt_mask)
    oracle_idx = int(np.argmax(ious)); sam_choice_idx = int(np.argmax(sam_scores))
    return {"iou_oracle": ious[oracle_idx], "iou_by_sam_score": ious[sam_choice_idx], "detected": True}

print("Canonical CLIP+SAM baseline functions loaded.")


Canonical CLIP+SAM baseline functions loaded.


## 3. Canonical Grounding DINO + SAM Pipeline (Primary)

In [11]:
def _contains_word_fuzzy(text, target_word, max_dist=1):
    def edit_distance(a, b):
        if abs(len(a) - len(b)) > max_dist + 1:
            return max_dist + 1
        prev = list(range(len(b) + 1))
        for i, ca in enumerate(a, 1):
            curr = [i] + [0] * len(b)
            for j, cb in enumerate(b, 1):
                curr[j] = min(prev[j] + 1, curr[j-1] + 1, prev[j-1] + (ca != cb))
            prev = curr
        return prev[-1]
    words = text.lower().replace(".", "").split()
    return any(edit_distance(w, target_word) <= max_dist for w in words)

def select_best_box(image, prompt, results):
    if len(results["boxes"]) == 0:
        return None
    boxes = results["boxes"].cpu().numpy()
    scores = results["scores"].cpu().numpy()
    has_left = _contains_word_fuzzy(prompt, "left")
    has_right = _contains_word_fuzzy(prompt, "right")
    if (has_left or has_right) and len(boxes) > 1:
        centers_x = (boxes[:, 0] + boxes[:, 2]) / 2
        candidate_idx = int(np.argmin(centers_x)) if has_left else int(np.argmax(centers_x))
        if scores[candidate_idx] > 0.10:
            return boxes[candidate_idx]
    return boxes[int(np.argmax(scores))]

def ground_with_dino(image, prompt, threshold=0.15, text_threshold=0.15):
    text = prompt.lower().strip()
    if not text.endswith("."):
        text += "."
    inputs = gd_processor(images=image, text=text, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = gd_model(**inputs)
    results = gd_processor.post_process_grounded_object_detection(
        outputs, inputs["input_ids"], threshold=threshold, text_threshold=text_threshold,
        target_sizes=[(image.height, image.width)]
    )[0]
    box = select_best_box(image, prompt, results)
    return box, results

def run_groundingdino_sam(image, prompt, gt_mask, predictor, threshold=0.15, text_threshold=0.15):
    box, raw_results = ground_with_dino(image, prompt, threshold, text_threshold)
    if box is None:
        return {"iou_oracle": 0.0, "iou_by_sam_score": 0.0, "detected": False, "box": None}
    predictor.set_image(np.array(image))
    masks, sam_scores, logits = predictor.predict(box=box, multimask_output=True)
    ious = score_masks_against_gt(masks, gt_mask)
    oracle_idx = int(np.argmax(ious)); sam_choice_idx = int(np.argmax(sam_scores))
    return {"iou_oracle": ious[oracle_idx], "iou_by_sam_score": ious[sam_choice_idx],
            "best_mask": masks[oracle_idx], "detected": True, "box": box}

print("Canonical Grounding DINO + SAM pipeline loaded.")


Canonical Grounding DINO + SAM pipeline loaded.


## 4. DINOv2 — Both Tested Roles

In [12]:
def get_dinov2_patch_features(pil_image):
    img_tensor = dinov2_transform(pil_image.convert("RGB")).unsqueeze(0).to(device)
    with torch.no_grad():
        features = dinov2.forward_features(img_tensor)
    patch_tokens = features["x_norm_patchtokens"].squeeze(0)
    return torch.nn.functional.normalize(patch_tokens, dim=-1)

def get_dinov2_mask_score(image, box, mask):
    dino_features = get_dinov2_patch_features(image)
    grid_size = int(round(dino_features.shape[0] ** 0.5))
    x0, y0, x1, y1 = box
    cx, cy = (x0 + x1) / 2, (y0 + y1) / 2
    cw, ch = (x1 - x0) * 0.4, (y1 - y0) * 0.4
    patch_w = image.width / grid_size
    patch_h = image.height / grid_size
    proto_feats = []
    for row in range(grid_size):
        for col in range(grid_size):
            px = (col + 0.5) * patch_w
            py = (row + 0.5) * patch_h
            if (cx - cw/2) <= px <= (cx + cw/2) and (cy - ch/2) <= py <= (cy + ch/2):
                proto_feats.append(dino_features[row * grid_size + col])
    if len(proto_feats) == 0:
        return 0.0
    prototype = torch.stack(proto_feats).mean(dim=0, keepdim=True)
    prototype = torch.nn.functional.normalize(prototype, dim=-1)
    sims = []
    for row in range(grid_size):
        for col in range(grid_size):
            px = int(min((col + 0.5) * patch_w, image.width - 1))
            py = int(min((row + 0.5) * patch_h, image.height - 1))
            if mask[py, px]:
                feat = dino_features[row * grid_size + col].unsqueeze(0)
                feat = torch.nn.functional.normalize(feat, dim=-1)
                sims.append((feat @ prototype.T).item())
    return float(np.mean(sims)) if sims else 0.0

def run_groundingdino_dino_sam(image, prompt, gt_mask, predictor):
    """Role B: Grounding DINO -> SAM (3 candidates) -> DINOv2 re-scoring."""
    box, raw_results = ground_with_dino(image, prompt)
    if box is None:
        return {"iou_selected": 0.0, "detected": False}
    predictor.set_image(np.array(image))
    masks, sam_scores, logits = predictor.predict(box=box, multimask_output=True)
    ious = score_masks_against_gt(masks, gt_mask)
    dino_scores = [get_dinov2_mask_score(image, box, m) for m in masks]
    dino_choice_idx = int(np.argmax(dino_scores))
    return {"iou_selected": ious[dino_choice_idx], "iou_sam_choice": ious[int(np.argmax(sam_scores))],
            "detected": True, "box": box}

def get_dinov2_refined_heatmap_v3(image, clip_heatmap, dinov2_features,
                                    clip_grid_size=7, dinov2_grid_size=37):
    """Role A: region-based CLIP+DINOv2 fusion (kept for completeness)."""
    flat_clip = clip_heatmap.flatten()
    weights = torch.softmax(flat_clip * 10, dim=0).to(dinov2_features.device)
    seed_features = []
    for idx in range(clip_grid_size * clip_grid_size):
        clip_row, clip_col = idx // clip_grid_size, idx % clip_grid_size
        frac_row = (clip_row + 0.5) / clip_grid_size
        frac_col = (clip_col + 0.5) / clip_grid_size
        dino_row = min(int(frac_row * dinov2_grid_size), dinov2_grid_size - 1)
        dino_col = min(int(frac_col * dinov2_grid_size), dinov2_grid_size - 1)
        dino_idx = dino_row * dinov2_grid_size + dino_col
        seed_features.append(dinov2_features[dino_idx])
    seed_features = torch.stack(seed_features)
    prototype = (weights.unsqueeze(1) * seed_features).sum(dim=0, keepdim=True)
    prototype = torch.nn.functional.normalize(prototype, dim=-1)
    dino_similarity = dinov2_features @ prototype.T
    dino_heatmap = dino_similarity.reshape(dinov2_grid_size, dinov2_grid_size).detach().cpu()
    clip_upsampled = torch.nn.functional.interpolate(
        clip_heatmap.unsqueeze(0).unsqueeze(0), size=(dinov2_grid_size, dinov2_grid_size),
        mode="bilinear", align_corners=False).squeeze()
    def min_max_norm(t):
        return (t - t.min()) / (t.max() - t.min() + 1e-8)
    return min_max_norm(clip_upsampled) * min_max_norm(dino_heatmap)

print("DINOv2 (both tested roles) functions loaded.")


DINOv2 (both tested roles) functions loaded.


## 5. Corruption Functions and Severity Levels

In [13]:
def apply_gaussian_blur(image, radius=5.0):
    return image.filter(ImageFilter.GaussianBlur(radius=radius))

def apply_gaussian_noise(image, std=25):
    arr = np.array(image).astype(np.float32)
    noise = np.random.normal(0, std, arr.shape)
    return Image.fromarray(np.clip(arr + noise, 0, 255).astype(np.uint8))

def apply_brightness_shift(image, factor=0.3):
    return ImageEnhance.Brightness(image).enhance(factor)

CORRUPTION_FUNCS = {
    "blur": lambda img: apply_gaussian_blur(img, radius=5.0),
    "noise": lambda img: apply_gaussian_noise(img, std=25),
    "brightness": lambda img: apply_brightness_shift(img, factor=0.3),
}

SEVERITY_LEVELS = {
    "blur": {
        "light": lambda img: apply_gaussian_blur(img, radius=2.0),
        "moderate": lambda img: apply_gaussian_blur(img, radius=5.0),
        "severe": lambda img: apply_gaussian_blur(img, radius=8.0),
    },
    "noise": {
        "light": lambda img: apply_gaussian_noise(img, std=12),
        "moderate": lambda img: apply_gaussian_noise(img, std=25),
        "severe": lambda img: apply_gaussian_noise(img, std=45),
    },
    "brightness": {
        "light": lambda img: apply_brightness_shift(img, factor=0.6),
        "moderate": lambda img: apply_brightness_shift(img, factor=0.3),
        "severe": lambda img: apply_brightness_shift(img, factor=0.15),
    },
}

print("Corruption functions, CORRUPTION_FUNCS, and SEVERITY_LEVELS loaded.")


Corruption functions, CORRUPTION_FUNCS, and SEVERITY_LEVELS loaded.


## 6. Sanity Check

In [14]:
sample = dataset["validation"][0]
image = sample["image"]
target = sample["samples"][0]
gt_mask = maskUtils.decode(target["mask"])

r_gdino = run_groundingdino_sam(image, target["sentences"][0], gt_mask, predictor)
print("GDINO+SAM sanity IoU:", r_gdino["iou_oracle"], "(expect high, e.g. > 0.8)")

r_clip = run_clip_sam(image, target["sentences"][0], gt_mask, predictor)
print("CLIP+SAM sanity IoU:", r_clip["iou_oracle"])


GDINO+SAM sanity IoU: 0.9845787672931026 (expect high, e.g. > 0.8)
CLIP+SAM sanity IoU: 0.21515022070340312


---
# PRIORITY 1 — DINOv2 Under Corruption
**The only hypothesis (RQ4, corruption half) with zero existing data anywhere in this project.** Runs first so it completes even if nothing else does. Compares Grounding DINO+SAM against Grounding DINO+SAM+DINOv2 (mask re-scoring role) across clean and all three corrupted conditions. N=100 images, fully resumable via the chunk runner (chunk size 10).

In [15]:
N_DINO_CORRUPT = 100
CHUNK_DINO_CORRUPT = 10
DINO_CORRUPT_DIR = f"{RESULTS_DIR}/priority1_dinov2_corruption"

def process_image_dino_corrupt(img_idx):
    rows = []
    sample = dataset["validation"][img_idx]
    base_image = sample["image"]

    for obj_idx, target in enumerate(sample["samples"]):
        gt_mask = maskUtils.decode(target["mask"])
        category = target["category"]
        prompts_to_test = [("minimal", category)] + [("referring", e) for e in target["sentences"]]

        for condition_name, corruption_fn in [("clean", None)] + list(CORRUPTION_FUNCS.items()):
            image = corruption_fn(base_image) if corruption_fn else base_image

            for prompt_type, prompt in prompts_to_test:
                r_base = run_groundingdino_sam(image, prompt, gt_mask, predictor)
                iou_base = r_base.get("iou_by_sam_score", 0.0) if r_base["detected"] else 0.0

                r_dino = run_groundingdino_dino_sam(image, prompt, gt_mask, predictor)
                iou_dino = r_dino.get("iou_selected", 0.0) if r_dino["detected"] else 0.0

                rows.append({
                    "img_idx": img_idx, "obj_idx": obj_idx, "category": category,
                    "prompt_type": prompt_type, "condition": condition_name,
                    "iou_gdino_sam": iou_base, "iou_gdino_dino": iou_dino,
                })
    return rows

dino_corrupt_df = run_chunked("dinov2_corruption", N_DINO_CORRUPT, CHUNK_DINO_CORRUPT,
                                DINO_CORRUPT_DIR, process_image_dino_corrupt)
dino_corrupt_df.to_csv(f"{RESULTS_DIR}/dinov2_corruption_final.csv", index=False)

print(f"\nPRIORITY 1 complete: {len(dino_corrupt_df)} rows")
print(dino_corrupt_df.groupby("condition")[["iou_gdino_sam", "iou_gdino_dino"]].mean())
diff = dino_corrupt_df["iou_gdino_dino"] - dino_corrupt_df["iou_gdino_sam"]
dino_corrupt_df["delta"] = diff
print()
print(dino_corrupt_df.groupby("condition")["delta"].apply(
    lambda d: pd.Series({"dino_better": (d > 0.01).sum(), "sam_better": (d < -0.01).sum(), "tied": (d.abs() <= 0.01).sum()})
))


[dinov2_corruption] chunk 0-10 already done, skipping
[dinov2_corruption] chunk 10-20 already done, skipping
[dinov2_corruption] chunk 20-30 already done, skipping
[dinov2_corruption] chunk 30-40 already done, skipping
[dinov2_corruption] chunk 40-50 already done, skipping
[dinov2_corruption] chunk 50-60 already done, skipping
[dinov2_corruption] chunk 60-70 already done, skipping
[dinov2_corruption] chunk 70-80 already done, skipping
[dinov2_corruption] chunk 80-90 already done, skipping
[dinov2_corruption] chunk 90-100 already done, skipping

PRIORITY 1 complete: 2592 rows
            iou_gdino_sam  iou_gdino_dino
condition                                
blur             0.449826        0.433175
brightness       0.541713        0.514473
clean            0.529950        0.502403
noise            0.517500        0.487253

condition              
blur        dino_better     72
            sam_better     194
            tied           382
brightness  dino_better     48
            sam_b

---
# PRIORITY 2 — CLIP Localization Diagnostic (Expanded)
Reproduces and strengthens the CLIP hit-rate number (previously N=25) at N=150 for a more statistically solid published figure. Independent of SAM — checks only whether CLIP's raw heatmap points land inside the correct object.

In [16]:
N_DIAG = 150
CHUNK_DIAG = 25
DIAG_DIR = f"{RESULTS_DIR}/priority2_clip_diagnostic"

def process_image_diag(img_idx):
    rows = []
    sample = dataset["validation"][img_idx]
    image = sample["image"]

    for obj_idx, target in enumerate(sample["samples"]):
        gt_mask = maskUtils.decode(target["mask"]).astype(bool)
        prompts_to_test = [("minimal", target["category"])] + \
                           [("referring", expr) for expr in target["sentences"]]

        for prompt_type, prompt in prompts_to_test:
            heatmap = get_clip_heatmap(image, prompt)
            _, raw_points, _ = heatmap_to_points(image, heatmap, grid_size=7, top_k=5, max_dist_frac=1.0)

            hits = 0
            for x, y in raw_points:
                xi, yi = int(min(x, image.width - 1)), int(min(y, image.height - 1))
                if gt_mask[yi, xi]:
                    hits += 1

            rows.append({
                "img_idx": img_idx, "obj_idx": obj_idx, "prompt_type": prompt_type,
                "prompt_text": prompt, "points_in_gt": hits, "any_point_in_gt": hits > 0,
            })
    return rows

loc_df = run_chunked("clip_diagnostic", N_DIAG, CHUNK_DIAG, DIAG_DIR, process_image_diag)
loc_df.to_csv(f"{RESULTS_DIR}/clip_localization_diagnostic.csv", index=False)

print(f"\nPRIORITY 2 complete: {len(loc_df)} rows")
print("=== Fraction of prompts where at least one CLIP top-5 point lands inside GT mask ===")
print(loc_df.groupby("prompt_type")["any_point_in_gt"].mean())


[clip_diagnostic] chunk 0-25 already done, skipping
[clip_diagnostic] chunk 25-50 already done, skipping
[clip_diagnostic] chunk 50-75 already done, skipping
[clip_diagnostic] chunk 75-100 already done, skipping
[clip_diagnostic] chunk 100-125 already done, skipping
[clip_diagnostic] chunk 125-150 already done, skipping

PRIORITY 2 complete: 970 rows
=== Fraction of prompts where at least one CLIP top-5 point lands inside GT mask ===
prompt_type
minimal      0.323194
referring    0.377652
Name: any_point_in_gt, dtype: float64


---
# PRIORITY 3 — Severity Sweep
Light/moderate/severe levels of each corruption type, consolidated into one self-contained results file. N=25, chunk size 5 (each image runs 9 corruption combinations).

In [17]:
N_SWEEP = 25
CHUNK_SWEEP = 5
SWEEP_DIR = f"{RESULTS_DIR}/priority3_severity_sweep"

def process_image_sweep(img_idx):
    rows = []
    sample = dataset["validation"][img_idx]
    clean_image = sample["image"]

    for obj_idx, target in enumerate(sample["samples"]):
        gt_mask = maskUtils.decode(target["mask"])
        category = target["category"]
        prompts_to_test = [("minimal", category)] + [("referring", e) for e in target["sentences"]]

        for corruption_name, severity_dict in SEVERITY_LEVELS.items():
            for severity_name, corruption_fn in severity_dict.items():
                corrupted_image = corruption_fn(clean_image)
                for prompt_type, prompt in prompts_to_test:
                    r = run_groundingdino_sam(corrupted_image, prompt, gt_mask, predictor)
                    iou = r.get("iou_by_sam_score", 0.0) if r["detected"] else 0.0
                    rows.append({
                        "img_idx": img_idx, "obj_idx": obj_idx, "category": category,
                        "prompt_type": prompt_type, "corruption_type": corruption_name,
                        "severity": severity_name, "iou": iou,
                    })
    return rows

sweep_df = run_chunked("severity_sweep", N_SWEEP, CHUNK_SWEEP, SWEEP_DIR, process_image_sweep)
sweep_df.to_csv(f"{RESULTS_DIR}/severity_sweep_final.csv", index=False)

print(f"\nPRIORITY 3 complete: {len(sweep_df)} rows")
severity_order = ["light", "moderate", "severe"]
summary = sweep_df.groupby(["corruption_type", "severity"])["iou"].agg(["mean", "median", "count"])
summary = summary.reindex(pd.MultiIndex.from_product(
    [SEVERITY_LEVELS.keys(), severity_order], names=["corruption_type", "severity"]))
print(summary)


[severity_sweep] chunk 0-5 already done, skipping
[severity_sweep] chunk 5-10 already done, skipping
[severity_sweep] chunk 10-15 already done, skipping
[severity_sweep] chunk 15-20 already done, skipping
[severity_sweep] chunk 20-25 already done, skipping

PRIORITY 3 complete: 1296 rows
                              mean    median  count
corruption_type severity                           
blur            light     0.707130  0.938404    144
                moderate  0.585926  0.841094    144
                severe    0.485135  0.537898    144
noise           light     0.692394  0.932018    144
                moderate  0.677477  0.892738    144
                severe    0.592330  0.761365    144
brightness      light     0.692022  0.954007    144
                moderate  0.702302  0.954325    144
                severe    0.674671  0.923281    144


---
# PRIORITY 4 — DINOv2 Role B on Clean Images (Consolidation)
Equivalent data already exists from an earlier successful Kaggle run today (N=100, clean only). Re-run here purely for a self-contained, reproducible results file alongside everything else. Lowest priority of the four experiment sections.

In [18]:
N_DINO_CLEAN = 100
CHUNK_DINO_CLEAN = 20
DINO_CLEAN_DIR = f"{RESULTS_DIR}/priority4_dinov2_clean"

def process_image_dino_clean(img_idx):
    rows = []
    sample = dataset["validation"][img_idx]
    image = sample["image"]
    num_objects = len(sample["samples"])

    for obj_idx, target in enumerate(sample["samples"]):
        gt_mask = maskUtils.decode(target["mask"])
        category = target["category"]
        prompts_to_test = [("minimal", category)] + [("referring", e) for e in target["sentences"]]

        for prompt_type, prompt in prompts_to_test:
            r = run_groundingdino_dino_sam(image, prompt, gt_mask, predictor)
            rows.append({
                "img_idx": img_idx, "obj_idx": obj_idx, "category": category,
                "prompt_type": prompt_type, "multi_instance": num_objects > 1,
                "gdino_sam_selected": r.get("iou_sam_choice", 0.0) if r["detected"] else 0.0,
                "gdino_dino_selected": r.get("iou_selected", 0.0) if r["detected"] else 0.0,
            })
    return rows

dino_clean_df = run_chunked("dinov2_clean", N_DINO_CLEAN, CHUNK_DINO_CLEAN, DINO_CLEAN_DIR, process_image_dino_clean)
dino_clean_df.to_csv(f"{RESULTS_DIR}/dinov2_rescore_results.csv", index=False)

print(f"\nPRIORITY 4 complete: {len(dino_clean_df)} rows")
print(dino_clean_df[["gdino_sam_selected", "gdino_dino_selected"]].agg(["mean", "median"]))
diff = dino_clean_df["gdino_dino_selected"] - dino_clean_df["gdino_sam_selected"]
print(f"DINOv2 better: {(diff > 0.01).sum()} | SAM-choice better: {(diff < -0.01).sum()} | Tied: {(diff.abs() <= 0.01).sum()}")


[dinov2_clean] chunk 0-20 already done, skipping
[dinov2_clean] chunk 20-40 already done, skipping
[dinov2_clean] chunk 40-60 already done, skipping
[dinov2_clean] chunk 60-80 already done, skipping
[dinov2_clean] chunk 80-100 already done, skipping

PRIORITY 4 complete: 648 rows
        gdino_sam_selected  gdino_dino_selected
mean              0.529950             0.502403
median            0.680958             0.610205
DINOv2 better: 53 | SAM-choice better: 211 | Tied: 384


---
# Main Comparison — Full Clean vs. Corrupted Experiment (CLIP+SAM vs. Grounding DINO+SAM)
**This is the paper's headline comparison, run here so this notebook is fully self-contained** — no need to reference results from any other session. Full 1,190-image dataset, all 4 conditions (clean/blur/noise/brightness), both pipelines, reporting both selected and oracle IoU. This is the single most expensive section in the notebook, which is why it's placed last: if the overnight run gets interrupted, the four priority sections above (the genuinely new evidence) are guaranteed to finish first regardless of what happens here. Fully resumable via the same chunk runner as everything else.

In [19]:
RUN_OPTIONAL_MAIN_EXPERIMENT = True  # keep ON: this notebook should be self-contained and include the main comparison itself

if RUN_OPTIONAL_MAIN_EXPERIMENT:
    N_MAIN = len(dataset["validation"])
    CHUNK_MAIN = 50
    MAIN_DIR = f"{RESULTS_DIR}/optional_main_experiment"

    CONDITIONS = [("clean", None)] + list(CORRUPTION_FUNCS.items())

    for condition_name, corruption_fn in CONDITIONS:
        def process_image_main(img_idx, _fn=corruption_fn, _cond=condition_name):
            rows = []
            sample = dataset["validation"][img_idx]
            base_image = sample["image"]
            image = _fn(base_image) if _fn else base_image
            num_objects = len(sample["samples"])

            for obj_idx, target in enumerate(sample["samples"]):
                gt_mask = maskUtils.decode(target["mask"])
                category = target["category"]
                prompts_to_test = [("minimal", category)] + [("referring", e) for e in target["sentences"]]

                for prompt_type, prompt in prompts_to_test:
                    r_gd = run_groundingdino_sam(image, prompt, gt_mask, predictor)
                    iou_gd = r_gd.get("iou_by_sam_score", 0.0) if r_gd["detected"] else 0.0
                    iou_gd_oracle = r_gd.get("iou_oracle", 0.0) if r_gd["detected"] else 0.0

                    r_clip = run_clip_sam(image, prompt, gt_mask, predictor)
                    iou_clip = r_clip.get("iou_by_sam_score", 0.0)
                    iou_clip_oracle = r_clip.get("iou_oracle", 0.0)

                    rows.append({
                        "img_idx": img_idx, "obj_idx": obj_idx, "category": category,
                        "prompt_type": prompt_type, "multi_instance": num_objects > 1,
                        "condition": _cond,
                        "iou_gdino_sam": iou_gd, "iou_gdino_sam_oracle": iou_gd_oracle,
                        "iou_clip_sam": iou_clip, "iou_clip_sam_oracle": iou_clip_oracle,
                    })
            return rows

        print(f"\n=== OPTIONAL: condition {condition_name} ===", flush=True)
        cond_df = run_chunked(f"main_{condition_name}", N_MAIN, CHUNK_MAIN, MAIN_DIR, process_image_main)
        cond_df.to_csv(f"{MAIN_DIR}/{condition_name}_ALL.csv", index=False)
        print(f"=== condition {condition_name} complete: {len(cond_df)} rows ===", flush=True)

    print("\nOptional main experiment complete.")
else:
    print("Optional main experiment SKIPPED (RUN_OPTIONAL_MAIN_EXPERIMENT=False). "
          "Existing full-scale results from today's earlier run remain the source for selected-IoU numbers.")



=== OPTIONAL: condition clean ===
[main_clean] chunk 0-50 already done, skipping
[main_clean] chunk 50-100 already done, skipping
[main_clean] chunk 100-150 already done, skipping
[main_clean] chunk 150-200 already done, skipping
[main_clean] chunk 200-250 already done, skipping
[main_clean] chunk 250-300 already done, skipping
[main_clean] chunk 300-350 already done, skipping
[main_clean] chunk 350-400 already done, skipping
[main_clean] chunk 400-450 already done, skipping
[main_clean] chunk 450-500 already done, skipping
[main_clean] chunk 500-550 already done, skipping
[main_clean] chunk 550-600 already done, skipping
[main_clean] chunk 600-650 already done, skipping
[main_clean] chunk 650-700 already done, skipping
[main_clean] chunk 700-750 already done, skipping
[main_clean] chunk 750-800 already done, skipping
[main_clean] chunk 800-850 already done, skipping
[main_clean] chunk 850-900 already done, skipping
[main_clean] chunk 900-950 already done, skipping
[main_clean] chunk 

## Final Summary (run any time, reads whatever has completed so far)

In [20]:
import glob

print("=== Files present in ./results ===")
for f in sorted(glob.glob(f"{RESULTS_DIR}/**/*.csv", recursive=True)):
    try:
        n = len(pd.read_csv(f))
        print(f"{f}  ({n} rows)")
    except Exception as e:
        print(f"{f}  (could not read: {e})")


=== Files present in ./results ===
./results/clip_localization_diagnostic.csv  (970 rows)
./results/dinov2_corruption_final.csv  (2592 rows)
./results/dinov2_rescore_results.csv  (648 rows)
./results/optional_main_experiment/blur_ALL.csv  (7678 rows)
./results/optional_main_experiment/brightness_ALL.csv  (7678 rows)
./results/optional_main_experiment/clean_ALL.csv  (7678 rows)
./results/optional_main_experiment/main_blur_0_50.csv  (289 rows)
./results/optional_main_experiment/main_blur_1000_1050.csv  (307 rows)
./results/optional_main_experiment/main_blur_100_150.csv  (322 rows)
./results/optional_main_experiment/main_blur_1050_1100.csv  (347 rows)
./results/optional_main_experiment/main_blur_1100_1150.csv  (362 rows)
./results/optional_main_experiment/main_blur_1150_1190.csv  (277 rows)
./results/optional_main_experiment/main_blur_150_200.csv  (316 rows)
./results/optional_main_experiment/main_blur_200_250.csv  (291 rows)
./results/optional_main_experiment/main_blur_250_300.csv  (324 